# Análisis de Ventas y Modelado Predictivo de Demanda

Este cuaderno contiene el análisis exploratorio de datos (EDA) y el entrenamiento de un modelo de Machine Learning para predecir la cantidad de productos vendidos a partir del tipo de producto y su precio unitario.

### Objetivos:
1. Cargar y explorar los datos de ventas procesados en Parquet.
2. Visualizar patrones clave en las ventas.
3. Desarrollar un modelo de regresión (Random Forest) para estimar la demanda (cantidad).
4. Exportar el modelo entrenado para su uso interactivo en la aplicación de Streamlit.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pickle
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score

# Configuración de visualizaciones
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Carga de Datos Procesados
Cargamos el archivo Parquet procesado por el pipeline de PySpark.

In [ ]:
# Nota: Si el pipeline de PySpark no se ha ejecutado, cargamos el CSV directamente como respaldo
data_path_parquet = "data/ventas_procesadas"
data_path_csv = "data/ventas.csv"

if os.path.exists(data_path_parquet):
    print("Cargando datos desde Parquet (procesado con PySpark)...")
    df = pd.read_parquet(data_path_parquet)
else:
    print("Cargando datos desde CSV (respaldo)...")
    df = pd.read_csv(data_path_csv)
    df['total_venta'] = df['cantidad'] * df['precio']

df.head()

## 2. Análisis Exploratorio y Visualización
Veamos un resumen estadístico y algunas gráficas clave.

In [ ]:
print("Resumen Estadístico:")
print(df.describe())

print("\nTotal de ingresos por producto:")
ingresos_prod = df.groupby('producto')['total_venta'].sum().sort_values(ascending=False)
print(ingresos_prod)

In [ ]:
# Gráfico de barras de ventas totales por producto
plt.figure(figsize=(10, 5))
sns.barplot(x=ingresos_prod.index, y=ingresos_prod.values, palette="viridis")
plt.title("Ingresos Totales por Producto")
plt.xlabel("Producto")
plt.ylabel("Ventas Totales ($)")
plt.xticks(rotation=45)
plt.tight_layout()
os.makedirs("data", exist_ok=True)
plt.savefig("data/ingresos_por_producto.png")
plt.show()

## 3. Preparación de Datos para Machine Learning
Queremos entrenar un modelo para predecir la **cantidad** vendida basada en el **producto** y su **precio**. Esto simula una curva de demanda del mercado.

In [ ]:
# Codificar la variable categórica 'producto'
le = LabelEncoder()
df['producto_encoded'] = le.fit_transform(df['producto'])

# Características (X) y Objetivo (y)
X = df[['producto_encoded', 'precio']]
y = df['cantidad']

# División en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Dimensiones de entrenamiento: {X_train.shape}")
print(f"Dimensiones de prueba: {X_test.shape}")

## 4. Entrenamiento del Modelo (Random Forest Regressor)
Entrenamos un modelo de bosque aleatorio para capturar relaciones no lineales en el precio y la demanda.

In [ ]:
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predicciones
y_pred = model.predict(X_test)

# Evaluación
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Error Cuadrático Medio (MSE): {mse:.4f}")
print(f"Coeficiente de Determinación (R^2): {r2:.4f}")

## 5. Exportar el Modelo
Guardamos el modelo entrenado y el `LabelEncoder` para poder utilizarlos en la aplicación interactiva de Streamlit (`app.py`).

In [ ]:
# Crear directorio de modelos si no existe
os.makedirs("models", exist_ok=True)

# Guardar modelo y codificador
model_data = {
    'model': model,
    'label_encoder': le,
    'products_info': df.groupby('producto')['precio'].mean().to_dict()
}

with open("models/sales_model.pkl", "wb") as f:
    pickle.dump(model_data, f)

print("¡Modelo y codificador de etiquetas guardados con éxito en 'models/sales_model.pkl'!")